## **E**xtract

Extraia a lista de IDs de usuário a partir do arquivo CSV. Para cada ID, faça uma requisição GET para obter os dados do usuário correspondente.

In [95]:
import pandas as pd

df = pd.read_csv('DTED.csv')
user_ids = df['UserID'].tolist()
print(user_ids)

[1, 2, 3, 4, 5]


In [96]:
import json

users = df.to_dict('records')
print(json.dumps(users, indent=2))

[
  {
    "UserID": 1,
    "name": "Naruto",
    "news": NaN
  },
  {
    "UserID": 2,
    "name": "Inata",
    "news": NaN
  },
  {
    "UserID": 3,
    "name": "Sasuke",
    "news": NaN
  },
  {
    "UserID": 4,
    "name": "Sakura",
    "news": NaN
  },
  {
    "UserID": 5,
    "name": "Ino",
    "news": NaN
  }
]


## **T**ransform

Utilize a API do Groq para gerar uma mensagem de saude nutrional personalizada para cada usuário.

In [25]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.1 MB/s eta 0:00:00


In [26]:
# Para gerar uma API Key:
# 1. Crie uma conta na Groq
# 2. Acesse a seção "API Keys"
# 3. Clique em "Create API Key"

# Substitua o texto TODO por sua API Key da OpenAI, ela será salva como uma variável de ambiente.
groq_api_key = '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'

In [97]:
from groq import Groq
import os

# Configure sua API key (pegue em https://console.groq.com/keys)
os.environ["GROQ_API_KEY"] = groq_api_key

# Cria o cliente (similar ao da OpenAI)
client = Groq()

def generate_ai_news(user_name):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # Modelo gratuito e rápido
        messages=[
            {
                "role": "system",
                "content": "Você é um especialista em nutricionismo."
            },
            {
                "role": "user",
                "content": f"Crie uma mensagem para {user_name} sobre a importância da boa alimentação (máximo de 100 caracteres)"
            }
        ],
        temperature=0.7,
        max_completion_tokens=500
    )
    return response.choices[0].message.content.strip('\"')

for user in users:
  news = generate_ai_news(user['name'])
  print(news)
  user['news'] = []
  user['news'].append({"description": news})

Naruto, comida saudável = energia para lutar!
Inata, alimente-se bem! Nutrição é vida.
Sasuke, coma bem!
Sakura, alimente-se bem!
Ino, alimente-se bem!


## **L**oad

Atualize a lista de "news" de cada usuário no arquivo com a nova mensagem gerada.

In [102]:
def salvar_no_csv(users, arquivo_csv='DTED.csv'):
    """
    Salva a lista users de volta no CSV, sobrescrevendo o arquivo

    Args:
        users: Lista de dicionários com os dados
        arquivo_csv: Nome do arquivo CSV (padrão: 'DTED.csv')
    """
    import pandas as pd
    import json

    # Converte para DataFrame
    df = pd.DataFrame(users)

    # Se coluna 'news' existe e é lista/dicionário, converte para string
    if 'news' in df.columns:
        df['news'] = df['news'].apply(lambda x: json.dumps(x, ensure_ascii=False) if x else '[]')

    # Sobrescreve o arquivo
    df.to_csv(arquivo_csv, index=False)
    print(f"✅ {arquivo_csv} atualizado com {len(users)} registros")

# Uso
salvar_no_csv(users, 'DTED.csv')

# Verificar se salvou corretamente
df_verificacao = pd.read_csv('DTED.csv')
print(f"Linhas no arquivo atualizado: {len(df_verificacao)}")
print(f"Colunas: {list(df_verificacao.columns)}")

✅ DTED.csv atualizado com 5 registros
Linhas no arquivo atualizado: 5
Colunas: ['UserID', 'name', 'news']


In [103]:
!git status

fatal: not a git repository (or any of the parent directories): .git
